# Ranking Results Explorer

Interactive exploration of scoring session results for a single batch.

Covers:
- Rankings between birds and between songs
- Ranking consistency per bird / per song
- Within-scorer vs. across-scorer consistency
- Scoring-mode comparison (`same_tutor` vs `all`)

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tables

# Make sure analyze_rankings is importable from the repo root
REPO = Path("E:/scoring")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import analyze_rankings as ar

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## Configuration

Set the batch and scoring mode here. Re-run all cells after changing.

In [ ]:
BATCH_DIR    = REPO / "batches" / "bk37wh86_rd75wh72_20260414"
SCORING_MODE = "same_tutor"   # "same_tutor", "all", or None for legacy root
TRAIT        = "repeat_propensity"
ELO_K        = 32.0

## Load batch metadata from HDF5

In [ ]:
uid_meta = {}
with tables.open_file(str(BATCH_DIR / "batch.h5"), mode="r") as h5:
    pairing = json.loads(h5.root.config._v_attrs["pairing"])
    for row in h5.root.manifest.iterrows():
        uid = row["uid"].decode()
        uid_meta[uid] = {
            "bird_id":         row["bird_id"].decode(),
            "role":            row["role"].decode(),
            "spec_idx":        int(row["spec_idx"]),
            "source_file":     row["source_file"].decode(),
            "snippet_start_s": float(row["snippet_start_s"]),
        }

print(f"Pairing:  {pairing['nest_father']} x {pairing['genetic_father']}")
print(f"Total UIDs in batch: {len(uid_meta)}")

meta_df = pd.DataFrame(uid_meta).T.reset_index().rename(columns={"index": "uid"})
meta_df.groupby("role")[["bird_id"]].nunique().rename(columns={"bird_id": "n_birds"})

## Load sessions

In [ ]:
sessions_dir = (BATCH_DIR / "sessions" / SCORING_MODE
                if SCORING_MODE else BATCH_DIR / "sessions")
sessions = ar.load_sessions(sessions_dir, trait=TRAIT)
n_fast   = ar.flag_fast_rounds(sessions)

scorers     = sorted({s["scorer"] for s in sessions})
total_rounds = sum(len(s["rounds"]) for s in sessions)

print(f"Scoring mode: {SCORING_MODE}")
print(f"Trait:        {TRAIT}")
print(f"Scorers:      {scorers}")
print(f"Sessions:     {len(sessions)}")
print(f"Total rounds: {total_rounds}  ({n_fast} flagged as too-fast)")

# Per-scorer breakdown
rows = []
for s in sessions:
    rnds = s["rounds"]
    n_ranked = sum(len(r["ranking"]) for r in rnds)
    n_flagged = sum(len(r["flagged"]) for r in rnds)
    rows.append({
        "scorer":   s["scorer"],
        "session":  Path(sessions_dir / f"{s['scorer']}_{TRAIT}").name,
        "rounds":   len(rnds),
        "ranked":   n_ranked,
        "flagged":  n_flagged,
    })
pd.DataFrame(rows)

## Compute Elo scores

In [ ]:
noise_uids = ar.collect_flagged_uids(sessions)
print(f"Noise/call UIDs excluded: {len(noise_uids)}")

elo_scores = ar.compute_elo(sessions, set(uid_meta), k=ELO_K,
                             excluded_uids=noise_uids)
bird_avgs  = ar.bird_averages(elo_scores, uid_meta)

# Per-snippet table
snippet_df = pd.DataFrame([
    {"uid": uid, "elo": score,
     "bird_id": uid_meta.get(uid, {}).get("bird_id", ""),
     "role":    uid_meta.get(uid, {}).get("role", "")}
    for uid, score in sorted(elo_scores.items(), key=lambda x: -x[1])
])
snippet_df.head(10)

## Bird rankings

Mean Elo per bird, sorted within each role group.

In [ ]:
bird_df = pd.DataFrame(bird_avgs).T
bird_df["mean_elo"] = bird_df["mean_elo"].astype(float)
bird_df["sd_elo"]   = bird_df["sd_elo"].astype(float)

role_order = [r for r in ar.ROLE_ORDER if r in bird_df["role"].values]
bird_df["role_rank"] = bird_df["role"].map(
    {r: i for i, r in enumerate(ar.ROLE_ORDER)}
).fillna(99)
bird_df = bird_df.sort_values(["role_rank", "mean_elo"], ascending=[True, False])

display(bird_df[["bird_id", "role", "mean_elo", "sd_elo", "n_snippets"]].reset_index(drop=True))

In [ ]:
output_stem = f"{BATCH_DIR.name}_{SCORING_MODE}" if SCORING_MODE else BATCH_DIR.name
ar.plot_bird_elo(bird_avgs, TRAIT, REPO / "results", output_stem)
plt.show()

## Song rankings

Per-snippet Elo by role. Each dot is one song snippet. Median per role shown as a horizontal bar.

In [ ]:
# Role-level summary
role_summary = snippet_df.groupby("role")["elo"].agg(
    n="count", median="median", mean="mean", sd="std"
).round(1)
display(role_summary)

ar.plot_snippet_elo(elo_scores, uid_meta, TRAIT, REPO / "results", output_stem)
plt.show()

## Ranking consistency

For each snippet, we track its position across every (scorer, round) it appeared in.

**Normalised rank**: 0 = ranked first (most of trait), 1 = ranked last.

- **Mean norm rank**: where the snippet tends to land in the ordering
- **SD norm rank**: how consistently it was placed (lower = more consistent)

Snippets that only appeared once have no SD (shown as × in the scatter).

In [ ]:
consistency = ar.compute_rank_consistency(sessions, excluded_uids=noise_uids)

cons_rows = []
for uid, info in consistency.items():
    meta = uid_meta.get(uid, {})
    npos = info["norm_positions"]
    cons_rows.append({
        "uid":            uid,
        "bird_id":        meta.get("bird_id", ""),
        "role":           meta.get("role", ""),
        "n_appearances":  info["n_appearances"],
        "n_scorers":      info["n_scorers"],
        "mean_norm_rank": round(np.mean(npos), 3) if npos else None,
        "sd_norm_rank":   round(np.std(npos),  3) if len(npos) > 1 else None,
        "scorers":        "; ".join(sorted(info["scorers"])),
    })

cons_df = pd.DataFrame(cons_rows).sort_values("mean_norm_rank")

print(f"Snippets with >=2 appearances (SD computable): "
      f"{(cons_df['n_appearances'] >= 2).sum()} / {len(cons_df)}")
print(f"Snippets seen by both scorers: "
      f"{(cons_df['n_scorers'] >= 2).sum()}")
display(cons_df.head(15))

In [ ]:
# Per-bird consistency: mean of each bird's snippet SDs
multi_cons = cons_df[cons_df["sd_norm_rank"].notna()].copy()
if not multi_cons.empty:
    bird_cons = (multi_cons
                 .groupby(["bird_id", "role"])["sd_norm_rank"]
                 .agg(n_snippets="count", mean_sd="mean", max_sd="max")
                 .round(3)
                 .reset_index()
                 .sort_values("mean_sd"))
    print("Per-bird ranking consistency (lower SD = more consistent):")
    display(bird_cons)
else:
    print("No snippets with multiple appearances yet — more scoring rounds needed for within-uid consistency.")

In [ ]:
ar.plot_rank_consistency(consistency, uid_meta, TRAIT, REPO / "results", output_stem)
plt.show()

## Inter-rater reliability

Kendall's τ-b between each pair of scorers on their shared UIDs.

τ = 1 → perfect agreement, 0 → no agreement, −1 → perfect disagreement.

In [ ]:
irr_rows    = ar.compute_irr(sessions)
scorer_rnks = ar.scorer_ranking(sessions, [])

irr_df = pd.DataFrame(irr_rows)
display(irr_df)

In [ ]:
ar.plot_scorer_agreement(scorer_rnks, uid_meta, irr_rows, TRAIT,
                          REPO / "results", output_stem)
plt.show()

### Per-scorer mean rank table

Mean rank assigned by each scorer to each shared UID, pivoted wide.
Lower rank = scorer placed it higher (more of trait).

In [ ]:
shared_uids = set.intersection(*[set(scorer_rnks[s]) for s in scorers]) if scorers else set()

if shared_uids:
    rank_rows = []
    for uid in sorted(shared_uids):
        meta = uid_meta.get(uid, {})
        row  = {"uid": uid[:8] + "...",
                "bird_id": meta.get("bird_id", ""),
                "role":    meta.get("role", "")}
        for sc in scorers:
            row[f"{sc}_mean_rank"] = round(scorer_rnks[sc].get(uid, float("nan")), 2)
        rank_rows.append(row)

    rank_df = pd.DataFrame(rank_rows)
    # Sort by average rank across scorers
    rank_cols = [f"{sc}_mean_rank" for sc in scorers]
    rank_df["avg_rank"] = rank_df[rank_cols].mean(axis=1)
    rank_df = rank_df.sort_values("avg_rank")
    display(rank_df.drop(columns="avg_rank").reset_index(drop=True))
else:
    print("No UIDs shared across all scorers.")

## Within-scorer vs. across-scorer consistency

**Within-scorer**: For UIDs that one scorer saw multiple times (across their sessions),
how consistent were their placements?

**Across-scorer**: For UIDs seen by multiple different scorers, how much do their
mean placements diverge?

In [ ]:
# Within-scorer consistency: compute per scorer separately
within_results = {}
for sess in sessions:
    sc = sess["scorer"]
    if sc not in within_results:
        within_results[sc] = ar.compute_rank_consistency([sess],
                                                          excluded_uids=noise_uids)
    else:
        # Merge additional sessions from same scorer
        extra = ar.compute_rank_consistency([sess], excluded_uids=noise_uids)
        for uid, info in extra.items():
            if uid in within_results[sc]:
                within_results[sc][uid]["positions"]      += info["positions"]
                within_results[sc][uid]["norm_positions"] += info["norm_positions"]
                within_results[sc][uid]["scorers"]        |= info["scorers"]
            else:
                within_results[sc][uid] = info
        for uid in within_results[sc]:
            within_results[sc][uid]["n_appearances"] = len(within_results[sc][uid]["positions"])
            within_results[sc][uid]["n_scorers"]     = len(within_results[sc][uid]["scorers"])

print("Within-scorer: snippets with >1 appearance per scorer")
for sc, cons in within_results.items():
    multi = {uid: d for uid, d in cons.items() if d["n_appearances"] > 1}
    if multi:
        sds = [np.std(d["norm_positions"]) for d in multi.values()
               if len(d["norm_positions"]) > 1]
        print(f"  {sc}: {len(multi)} re-seen snippets, "
              f"mean SD = {np.mean(sds):.3f}" if sds else f"  {sc}: {len(multi)} re-seen (SD not computable)")
    else:
        print(f"  {sc}: no snippets seen more than once within sessions")

In [ ]:
# Across-scorer consistency: for each UID, spread of mean ranks across scorers
across_rows = []
for uid in set.union(*[set(scorer_rnks[s]) for s in scorer_rnks]) if scorer_rnks else set():
    ranks_by_scorer = {sc: scorer_rnks[sc][uid]
                       for sc in scorers if uid in scorer_rnks[sc]}
    if len(ranks_by_scorer) < 2:
        continue
    meta = uid_meta.get(uid, {})
    vals = list(ranks_by_scorer.values())
    across_rows.append({
        "uid":       uid[:8] + "...",
        "bird_id":   meta.get("bird_id", ""),
        "role":      meta.get("role", ""),
        "n_scorers": len(ranks_by_scorer),
        "rank_range": round(max(vals) - min(vals), 2),
        "rank_sd":    round(float(np.std(vals)), 3),
        **{f"{sc}_rank": round(ranks_by_scorer.get(sc, float("nan")), 2)
           for sc in scorers},
    })

if across_rows:
    across_df = pd.DataFrame(across_rows).sort_values("rank_sd", ascending=False)
    print(f"Across-scorer disagreement (higher = less consistent):")
    print(f"  Mean SD across shared UIDs: {across_df['rank_sd'].mean():.3f}")
    print(f"  Top 10 most-disagreed-upon snippets:")
    display(across_df.head(10).reset_index(drop=True))
else:
    print("No UIDs scored by more than one scorer yet.")

## Scoring mode comparison: `same_tutor` vs `all`

If sessions exist in both modes, compare Elo and IRR side by side.
UIDs may differ between modes (different pools were presented).

In [ ]:
mode_results = {}
for mode in ["same_tutor", "all"]:
    sdir = BATCH_DIR / "sessions" / mode
    if not sdir.exists():
        continue
    sesses = ar.load_sessions(sdir, trait=TRAIT)
    if not sesses:
        continue
    ar.flag_fast_rounds(sesses)
    noise = ar.collect_flagged_uids(sesses)
    elos  = ar.compute_elo(sesses, set(uid_meta), k=ELO_K, excluded_uids=noise)
    bavgs = ar.bird_averages(elos, uid_meta)
    irr   = ar.compute_irr(sesses)
    mode_results[mode] = {
        "sessions": sesses,
        "n_scored": len(elos),
        "n_noise":  len(noise),
        "elos":     elos,
        "bird_avgs": bavgs,
        "irr":      irr,
    }
    scorers_m = sorted({s["scorer"] for s in sesses})
    tau_str = ", ".join(f"{r['scorer_a']} vs {r['scorer_b']}: tau={r['tau']}"
                        for r in irr) if irr else "(single scorer)"
    print(f"Mode: {mode}")
    print(f"  Scorers: {scorers_m}  |  Scored: {len(elos)} UIDs  |  IRR: {tau_str}")

In [ ]:
if len(mode_results) >= 2:
    # Compare per-bird Elo across modes for UIDs present in both
    all_birds = set.union(*[set(r["bird_avgs"]) for r in mode_results.values()])
    cmp_rows = []
    for bird_id in sorted(all_birds):
        row = {"bird_id": bird_id}
        for mode, res in mode_results.items():
            ba = res["bird_avgs"].get(bird_id)
            row[f"{mode}_mean_elo"] = ba["mean_elo"] if ba else None
            row["role"] = ba["role"] if ba else row.get("role", "")
        cmp_rows.append(row)
    cmp_df = pd.DataFrame(cmp_rows).sort_values("role")
    print("Per-bird mean Elo by scoring mode:")
    display(cmp_df.reset_index(drop=True))
else:
    print("Only one mode has data — comparison not available.")